# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thahsinj06/Fly-rank-ml-internship-work/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### Task type: Scoring

My lane is **Refresh / Content Opportunity Scoring**.

I frame this as a **scoring problem** because the goal is not only to say whether a page is good or bad. The useful output is a score that represents how strongly a page should be considered for review, so pages can be ranked from higher to lower refresh opportunity.

The score can support a content team's decision about **which pages to review first** for actions such as refreshing, expanding, protecting, pruning, or monitoring.

This is related to ranking because the final pages are ordered by their scores, but I am framing the underlying task as scoring because each page receives an opportunity score rather than only a rank.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### Target / proxy: observed decline as a refresh-opportunity proxy

For the starter dataset, I would use `is_declining_label` as a **proxy target**:

`is_declining_label = (trend_direction == "down")`

This label comes from an observed pattern in the current data window rather than a future outcome. It therefore represents **current evidence of decline**, not proof that refreshing the page will improve its performance.

I use this proxy because a stronger target would require a clearly defined future outcome, such as measuring whether a page's performance declines or recovers in a future time window.

For this assignment, the proxy lets me test whether observable page signals can help prioritise pages showing evidence of decline. The final output should therefore be treated as **directional decision-support**, not a guarantee of refresh impact.

In [12]:
import pandas as pd

url = "https://raw.githubusercontent.com/thahsinj06/Fly-rank-ml-internship-work/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print("Proxy target:", "is_declining_label")
print("\nClass counts:")
print(df["is_declining_label"].value_counts())

print("\nClass proportions:")
print(df["is_declining_label"].value_counts(normalize=True).round(3))

Proxy target: is_declining_label

Class counts:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64

Class proportions:
is_declining_label
1    0.542
0    0.458
Name: proportion, dtype: float64


### Success metric: Precision@K

I will use **Precision@K** as the main success metric.

Precision@K measures the proportion of the top K pages selected by the model that have the target proxy label. For example, Precision@20 asks: "Of the 20 highest-scored pages, how many are labelled as declining?"

This metric fits the business action because the model is intended to **prioritise a limited review queue**. A high Precision@K means that the pages placed near the top of the queue are more likely to show observed signs of decline.

I prefer this over overall accuracy because the goal is not to classify every page equally. The practical value comes from making the **highest-priority recommendations useful** to the content team.

In [13]:
# Define the review budget used for evaluating the top-ranked pages
k_values = [20, 50]

print("Evaluation metric: Precision@K")
print("Review budgets:", k_values)
print("Proxy-positive rate:", round(df["is_declining_label"].mean(), 3))


Evaluation metric: Precision@K
Review budgets: [20, 50]
Proxy-positive rate: 0.542


### Unit of analysis: one content page

The unit of analysis is **one content page**.

Each row represents a page and contains attributes describing its content, age, recent search performance, and trend. These page-level signals are used to identify pages that may deserve higher or lower refresh priority.

The important distinction is that the model is scoring **pages**, not individual search queries, users, or visits.

In [14]:
# Show the page-level unit of analysis
columns_to_show = [
    "content_type",
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count",
    "trend_direction",
    "is_declining_label"
]

unit_df = df[columns_to_show].head(10)

display(unit_df)

,content_type,content_age_days,days_since_last_update,impressions_90d,avg_position,ctr,word_count,trend_direction,is_declining_label
0,keyword article,187,20,3803,10.6,0.76,3221.0,down,1
1,keyword article,445,25,15320,20.3,0.05,2481.0,down,1
2,keyword article,141,20,12581,36.5,0.09,3515.0,down,1
3,keyword article,463,22,11751,6.2,0.49,NaN,stable,0
4,keyword article,263,14,19140,44.0,0.13,2803.0,down,1
5,keyword article,147,20,3970,8.5,0.03,3080.0,down,1
6,keyword article,90,20,20,7.0,0.00,3059.0,down,1
7,keyword article,445,22,1724,21.2,0.06,NaN,stable,0
8,keyword article,90,20,32574,46.0,0.09,3807.0,down,1
9,keyword article,257,104,1240,4.9,0.16,NaN,down,1


### Why ML may beat a fixed rule

A fixed rule could identify refresh candidates using a few thresholds, for example:

`if impressions are declining and the page is old → review`

The problem is that page performance depends on multiple signals at the same time. A page can be old but still perform well, while a newer page may already show signs of decline. Search position, impressions, CTR, content age, update recency, and content characteristics can also interact differently across pages.

A fixed rule requires manually choosing thresholds and may miss pages that show a weaker combination of several signals. A scoring model can learn patterns across these features and produce a more flexible priority score.

However, I would not assume that ML is automatically better. The ML approach should be compared against a simple fixed-rule baseline using the same **Precision@K** metric. If it does not improve prioritisation, the simpler rule may be preferable.

The intended output is therefore **decision-support**: a ranked list of pages for human review, not an automatic instruction to refresh every high-scoring page.

In [15]:
# Example of a simple fixed-rule baseline
# This is only a framing example, not the final baseline model.

rule_candidates = (
    (df["content_age_days"] > df["content_age_days"].median()) &
    (df["trend_direction"] == "down")
)

print("Pages selected by example fixed rule:", rule_candidates.sum())
print("Total pages:", len(df))
print("Selection rate:", round(rule_candidates.mean(), 3))


Pages selected by example fixed rule: 6551
Total pages: 30000
Selection rate: 0.218


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.